In [1]:
#| default_exp link.join

In [2]:
#| export
from __future__ import annotations

import logging

import pandas as pd

SEA_VALUE = "MSL_filtered_GIA_corrected_adjusted"
SEA_TREND = "trend_MSL_filtered_GIA_corrected_adjusted"
DROP_COLS = [
    "Disaster Type",
    "Disaster Subgroup",
    "Start Year", "Start Month", "Start Day",
    "End Year",   "End Month",   "End Day",
]

logger = logging.getLogger("myproj.link")

# Daten Joinen

## Setup

In [3]:
import numpy as np

from myproj.pipeline import run_import, run_pre_filter, run_cleaning, run_post_filter, run_transform

emdat_raw, sea_level_raw = run_import()
emdat, sea_level = run_pre_filter(emdat_raw, sea_level_raw)
emdat, sea_level = run_cleaning(emdat, sea_level)
emdat_flood = run_post_filter(emdat)
emdat_flood = run_transform(emdat_flood)


print(f"EMDAT nach Pipeline: {emdat.shape[0]:,} Zeilen, {emdat.shape[1]} Spalten")
print(f"Sea-Level nach Pipeline: {sea_level.shape[0]:,} Zeilen, {sea_level.shape[1]} Spalten")

2026-05-13 22:05:37 | myproj.pipeline      | INFO     | run_import | start
2026-05-13 22:05:37 | myproj.io            | INFO     | Lade Rohdatei: public_emdat_1991_2024.xlsx
2026-05-13 22:05:41 | myproj.io            | INFO     | load_raw_data | file: public_emdat_1991_2024.xlsx | rows: 20657 | cols: 47
2026-05-13 22:05:41 | myproj.io            | INFO     | Lade Rohdatei: omi_climate_sl_medsea_area_averaged_anomalies_19990220_P20250729.nc
2026-05-13 22:05:41 | myproj.io            | INFO     | load_raw_data | file: omi_climate_sl_medsea_area_averaged_anomalies_19990220_P20250729.nc | variables: 2 | dimensions: {'time': 9405}
2026-05-13 22:05:41 | myproj.pipeline      | INFO     | run_import | done
2026-05-13 22:05:41 | myproj.pipeline      | INFO     | run_pre_filter | start
2026-05-13 22:05:41 | myproj.transform     | INFO     | select_columns | cols: 47 → 18
2026-05-13 22:05:41 | myproj.transform     | INFO     | filter_to_sea_level_coverage | rows: 20657 → 16699 | removed before: 3

EMDAT nach Pipeline: 16,699 Zeilen, 18 Spalten
Sea-Level nach Pipeline: 9,405 Zeilen, 3 Spalten


## Sea-Level vorbereiten

`sea_level["time"]` muss als sortierter DatetimeIndex vorliegen, damit `.map()` und DatetimeIndex-Slicing im Join korrekt funktionieren.

In [4]:
#| export
def prepare_sea_level_ts(sea_level: pd.DataFrame, time_col: str = "time") -> pd.DataFrame:
    """Stellt sicher, dass die Sea-Level-Zeitreihe nach datetime sortiert vorliegt."""
    df = sea_level.copy()
    df[time_col] = pd.to_datetime(df[time_col])
    return df.sort_values(time_col).reset_index(drop=True)

In [5]:
sea_level_ts = prepare_sea_level_ts(sea_level)
print(
    f"Sea-Level: {len(sea_level_ts):,} Tageswerte | "
    f"{sea_level_ts['time'].min().date()} bis {sea_level_ts['time'].max().date()}"
)

Sea-Level: 9,405 Tageswerte | 1999-02-20 bis 2024-11-19


## Verknüpfung und Anreicherung

### Format-Entscheid

| Tabelle | Aktuelles Format | Transformation nötig? | Grund |
|---|---|---|---|
| `emdat_flood` | **Wide**, eine Zeile pro Flood-Ereignis | Nein | Spalten werden nur ergänzt |
| `sea_level_ts` | **Long / Zeitreihe**, eine Zeile pro Tag | Nein | Dient als Lookup-Tabelle |
| `flood_linked` (Output) | **Wide** | Nein | Ereignis und Sea-Level-Spalten, direkt analysierbar |

Beide Tabellen sind bereits im richtigen Format. Der Join hängt Sea-Level-Spalten an `emdat_flood` an. Keine Format-Transformation nötig.

### Left Join

`emdat_flood` ist die führende Tabelle. Sea-Level-Werte werden über `.map()` auf einen DatetimeIndex angehängt. Fehlt der genaue Tag im Sea-Level-Datensatz, entsteht NaN, erklärbar durch `start_date_quality`.

| Neue Spalte | Beschreibung |
|---|---|
| `sea_level_at_start` | Meeresspiegel am `start_date`, primäre Analysevariable |
| `sea_trend_at_start` | Langfristiger Trend am `start_date` |
| `sea_level_at_end` | Meeresspiegel am `end_date` |
| `mean_sea_level_while_disaster` | Mittlerer Meeresspiegel zwischen `start_date` und `end_date`. Sinnvoll bei langen Riverine-Floods, bei Flash-Floods (~1–3 Tage) ähnlich wie `sea_level_at_start` |


In [6]:
#| export
def _mean_sea_level(
    start: pd.Timestamp, end: pd.Timestamp, sea_msl_idx: pd.Series
) -> float:
    if pd.isna(start) or pd.isna(end) or end < start:
        return float("nan")
    return sea_msl_idx.loc[start:end].mean()

In [7]:
#| export
def join_sea_level(
    emdat_flood: pd.DataFrame,
    sea_level_ts: pd.DataFrame,
    sea_value: str = SEA_VALUE,
    sea_trend: str = SEA_TREND,
) -> pd.DataFrame:
    """Verknüpft EMDAT-Flood-Ereignisse mit Sea-Level-Messwerten."""
    df = emdat_flood.copy()
    sea_msl_idx   = sea_level_ts.set_index("time")[sea_value]
    sea_trend_idx = sea_level_ts.set_index("time")[sea_trend]

    df["sea_level_at_start"] = df["start_date"].map(sea_msl_idx)
    df["sea_trend_at_start"] = df["start_date"].map(sea_trend_idx)
    df["sea_level_at_end"]   = df["end_date"].map(sea_msl_idx)
    df["mean_sea_level_while_disaster"] = df.apply(
        lambda row: _mean_sea_level(row["start_date"], row["end_date"], sea_msl_idx),
        axis=1,
    )

    n_at_start = int(df["sea_level_at_start"].notna().sum())
    n_mean     = int(df["mean_sea_level_while_disaster"].notna().sum())
    logger.info(
        "join_sea_level | events: %d | sea_level_at_start_not_null: %d | mean_sea_level_not_null: %d",
        len(df), n_at_start, n_mean,
    )
    return df

In [8]:
emdat_flood = join_sea_level(emdat_flood, sea_level_ts)

2026-05-13 22:05:42 | myproj.link          | INFO     | join_sea_level | events: 292 | sea_level_at_start_not_null: 292 | mean_sea_level_not_null: 292


## Transformationen (Lags + Skalierung)
Hier werden die in `04_Transform.ipynb` definierten Lag- und Skalierungs-Funktionen angewendet.

In [9]:
from myproj.transform.transform import add_sea_level_lags, add_sea_level_z_scores

In [10]:
emdat_flood = add_sea_level_lags(emdat_flood, sea_level_ts)

2026-05-13 22:05:42 | myproj.transform     | INFO     | add_sea_level_lags | n_lags: 5 | rows: 292


In [11]:
emdat_flood = add_sea_level_z_scores(emdat_flood, sea_level_ts)

2026-05-13 22:05:42 | myproj.transform     | INFO     | add_sea_level_z_scores | msl_ref: μ=5.9631 σ=3.2936 cm | rows: 292


## Redundante Spalten entfernen

Nach Filter und Join sind einige Spalten redundant:

| Spalte | Grund |
|---|---|
| `Disaster Type` | Konstant "Flood" nach Filter |
| `Disaster Subgroup` | Konstant "Hydrological" nach Filter |
| `Start/End Year/Month/Day` | Durch `start_date`, `end_date` und `*_quality` ersetzt |

In [12]:
#| export
def drop_join_redundant_cols(
    df: pd.DataFrame, cols: list[str] = DROP_COLS
) -> pd.DataFrame:
    """Entfernt Spalten, die nach Filter und Join redundant sind."""
    result = df.drop(columns=cols).reset_index(drop=True)
    logger.info(
        "drop_join_redundant_cols | cols: %d → %d | dropped: %s",
        df.shape[1], result.shape[1], cols,
    )
    return result

In [13]:
flood_linked = drop_join_redundant_cols(emdat_flood)
print(f"Spalten nach Bereinigung: {flood_linked.shape[1]}")
print(flood_linked.columns.tolist())

2026-05-13 22:05:42 | myproj.link          | INFO     | drop_join_redundant_cols | cols: 36 → 28 | dropped: ['Disaster Type', 'Disaster Subgroup', 'Start Year', 'Start Month', 'Start Day', 'End Year', 'End Month', 'End Day']


Spalten nach Bereinigung: 28
['DisNo.', 'ISO', 'Country', 'Region', 'Disaster Subtype', 'Origin', 'Latitude', 'Longitude', 'Total Affected', 'Total Deaths', 'start_date', 'end_date', 'start_date_quality', 'end_date_quality', 'season', 'event_duration_days', 'has_coordinates', 'sea_level_at_start', 'sea_trend_at_start', 'sea_level_at_end', 'mean_sea_level_while_disaster', 'sea_level_lag_1d', 'sea_level_lag_2d', 'sea_level_lag_3d', 'sea_level_lag_4d', 'sea_level_lag_5d', 'sea_level_at_start_z', 'mean_sea_level_while_disaster_z']
